# MVP1 — Casos especiales y faltantes relevantes

**Objetivo:** documentar, de forma breve y reproducible, ejemplos que pueden afectar a la interpretación del análisis de notas de corte y cotizaciones.

La prioridad es la validez de las fuentes: se señalan la procedencia, la causa probable y la limitación, pero no se eliminan observaciones ni se presentan equivalencias provisionales como definitivas.

## Fuentes utilizadas

- `data/processed/SCRUM45_registro_atipicos.csv`: registro de 109 grados/campus con marcas y decisiones.
- `data/processed/SCRUM45_campos_subconjunto.csv`: resumen de 57 campos U-Ranking usados para revisar la sensibilidad.
- `data/processed/SCRUM42_borrador_por_grado_NO_VALIDADO.csv`: propuesta de correspondencia entre grado y campo.
- `docs/entregables/SCRUM45_atipicos.md` y `docs/fuentes/INFORME_FUENTES_Y_REVISION_GPR_2026-11-07.md`: contexto metodológico y advertencias de procedencia.

Los CSV están separados por `;` y codificados en UTF-8 con BOM.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def cargar(nombre):
    return pd.read_csv(ROOT / nombre, sep=";", encoding="utf-8-sig")

registro = cargar("data/processed/SCRUM45_registro_atipicos.csv")
campos = cargar("data/processed/SCRUM45_campos_subconjunto.csv")
mapeo = cargar("data/processed/SCRUM42_borrador_por_grado_NO_VALIDADO.csv")

anios = ["2022", "2023", "2024", "2025", "2026"]
print(f"Registro: {len(registro)} filas | Campos del subconjunto: {len(campos)}")
print(f"Filas aptas para el subconjunto: {registro['incluido_subconjunto'].astype(str).str.lower().eq('true').sum()}")

## 1. Faltantes por grados de nueva implantación

**Ejemplo:** hay grados con algún año sin nota de corte. La causa probable es que no existía oferta o serie publicada para ese año, no que la nota fuera cero.

**Procedencia:** series de notas originales recopiladas en `data/raw/grados_upv_notas_corte_2022_2026.csv` y `data/raw/grados_uv_notas_corte_2022_2026.csv`.

**Limitación:** no se imputa el valor faltante; las comparaciones temporales de esos grados deben usar solo los años disponibles.

In [ ]:
faltantes = registro[registro[anios].isna().any(axis=1)]
faltantes[["universidad", "grado_original", "campus", *anios, "procedencia_nota"]].head(10)

## 2. Nota mínima repetida

**Ejemplo:** el Doble Grado ADE + Turismo (UPV Alcoy) mantiene 5,00 en los cinco años. Es compatible con la nota mínima de acceso y no debe interpretarse automáticamente como una serie de demanda competitiva.

**Procedencia:** CSV original de notas de corte; la adjudicación oficial no está certificada en este paquete.

**Limitación:** se conserva como dato observado, pero conviene marcarlo en gráficos y evitar conclusiones fuertes sobre su evolución.

In [ ]:
nota_minima = registro[registro[anios].eq(5).all(axis=1)]
nota_minima[["universidad", "grado_original", "campus", *anios, "procedencia_nota"]]

## 3. Cambio interanual poco habitual

**Ejemplo:** Ingeniería en Tecnologías Industriales (UPV Vera) presenta la serie 11,60 → 10,88 → 12,07 → 10,80 → 11,00, con un salto máximo de 1,27 puntos.

**Causa probable:** cambio real de convocatoria, campus o criterio de publicación; también puede ser una incidencia de transcripción.

**Limitación:** el registro lo marca para revisión, pero no permite decidir cuál valor es incorrecto. Debe contrastarse con la fuente primaria antes de usarlo como tendencia.

In [ ]:
registro.sort_values("salto_max", ascending=False)[
    ["universidad", "grado_original", "campus", *anios, "salto_max", "procedencia_nota"]
].head(5)

## 4. Correspondencias de grado provisionales

**Ejemplo:** varias filas tienen la marca `U3`: el mapeo de `main` asigna un campo distinto al candidato de SCRUM-42. También hay equivalencias manuales que agrupan grados no idénticos.

**Causa probable:** coexistencia de dos tablas de mapeo elaboradas con criterios distintos.

**Limitación:** el campo nacional puede repetirse para varios grados/campus y no constituye una observación salarial independiente por fila. No debe calcularse una correlación final hasta validar el mapeo.

In [ ]:
u3 = registro[registro["marcas"].fillna("").str.contains("U3")]
u3[["grado_original", "campus", "campo_uranking_candidato", "campo_mapeo_main", "procedencia_cruce"]].head(8)

## 5. Valores salariales influyentes, pero no necesariamente erróneos

**Ejemplos:** Medicina aparece con 44.153 € y es el único campo con residuo estandarizado grande; Arquitectura técnica, Bioquímica e Ingeniería geomática influyen en el ajuste.

**Procedencia:** indicador nacional de U-Ranking 2024, base media anual de cotización en España para una cohorte que egresó en 2019–2020.

**Limitación:** no es el salario de recién graduados de UPV/UV ni de los admitidos en 2024. Los valores se mantienen y se señalan; influencia estadística no significa error de fuente.

In [ ]:
campos[campos["influyente"].astype(str).str.lower().eq("true")][
    ["campo_uranking_candidato", "nota_media_2024", "cotizacion_2024", "residuo_estandarizado", "distancia_cook", "decision"]
].sort_values("distancia_cook", ascending=False)

## Conclusión para el MVP1

Los casos deben presentarse como **advertencias de interpretación**, no como errores corregidos automáticamente. Las notas faltantes permanecen faltantes, las notas atípicas se revisan sin suprimirlas y las equivalencias de grado siguen siendo provisionales. La principal limitación es que las fuentes de notas y el mapeo grado–campo todavía no están completamente validados; por tanto, cualquier correlación debe considerarse exploratoria.